# A6: Naive RAG vs Contextual Retrieval

This notebook is my full A6 implementation.

Chapter source justification:
- My student ID is `st125982`, so the last digit is `2`.
- Based on the assignment rule, I must use **Chapter 2**.
- I used `2.pdf` in this repo as the Chapter 2 source text (Words and Tokens from the SLP material).

What is included:
1. Source discovery and data preparation
2. Naive RAG and Contextual Retrieval pipelines
3. ROUGE-1 / ROUGE-2 / ROUGE-L evaluation
4. Submission JSON generation in `answer/response-st-125982-chapter-2.json`

Models used:
- Retriever: `sentence-transformers/all-MiniLM-L6-v2`
- Generator: extractive sentence selector (local baseline)

In [4]:
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Tuple
import json
import re

import numpy as np
from pypdf import PdfReader
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings("ignore")


In [5]:
# Assignment configuration 
STUDENT_ID = "st125982"
CHAPTER_NO = 2
PDF_PATH = Path("2.pdf")
OUTPUT_DIR = Path("answer")


In [6]:
# QA pairs (20 required)
qa_pairs = [
    {
        "question": "What is tokenization in modern NLP?",
        "ground_truth_answer": "Tokenization is the process of separating running text into words, subwords, or other token units used by NLP systems.",
    },
    {
        "question": "Why was ELIZA important in NLP history?",
        "ground_truth_answer": "ELIZA showed that simple pattern matching over words could imitate conversation and highlighted early chatbot impacts on users.",
    },
    {
        "question": "How many words can be counted in the picnic sentence depending on punctuation treatment?",
        "ground_truth_answer": "The picnic sentence has 16 words when punctuation is excluded and 18 when punctuation is counted as separate words.",
    },
    {
        "question": "What are filled pauses in spoken language?",
        "ground_truth_answer": "Filled pauses are disfluencies such as 'uh' and 'um' that occur in speech and may still be useful features in speech processing.",
    },
    {
        "question": "What is the difference between word types and word instances?",
        "ground_truth_answer": "Word types are distinct vocabulary items in a corpus, while word instances are the total running occurrences of words.",
    },
    {
        "question": "Why might capitalization be kept or removed in NLP?",
        "ground_truth_answer": "It depends on the task: capitalization may be removed for normalization or retained when case carries useful information.",
    },
    {
        "question": "Why is defining a word harder in Chinese than in English?",
        "ground_truth_answer": "Chinese writing does not use spaces to mark word boundaries, so segmentation choices determine what counts as a word.",
    },
    {
        "question": "What is a morpheme?",
        "ground_truth_answer": "A morpheme is a meaningful subpart of a word, such as '-er' in 'longer'.",
    },
    {
        "question": "What is Byte-Pair Encoding (BPE) used for?",
        "ground_truth_answer": "BPE induces a subword vocabulary by merging frequent character sequences and is widely used for tokenization.",
    },
    {
        "question": "What role do regular expressions play in NLP pipelines?",
        "ground_truth_answer": "Regular expressions provide a formal way to specify and manipulate text patterns and are common preprocessing tools in NLP.",
    },
    {
        "question": "What does edit distance measure?",
        "ground_truth_answer": "Edit distance measures string similarity by counting insertions, deletions, and substitutions needed to transform one string into another.",
    },
    {
        "question": "How can disfluencies help speech recognition?",
        "ground_truth_answer": "Disfluencies can signal restarts and upcoming structure, so they may improve prediction in speech recognition systems.",
    },
    {
        "question": "Why might punctuation be treated as tokens in language models?",
        "ground_truth_answer": "Punctuation helps indicate boundaries and meaning cues, so many language models treat punctuation as separate tokens.",
    },
    {
        "question": "What is an utterance?",
        "ground_truth_answer": "An utterance is the spoken-language counterpart of a sentence.",
    },
    {
        "question": "How did Weizenbaum's ELIZA produce responses?",
        "ground_truth_answer": "ELIZA matched user text patterns and transformed them into scripted template responses.",
    },
    {
        "question": "Why can different Chinese segmentation standards produce different word counts?",
        "ground_truth_answer": "Different standards define word boundaries differently, including treatment of names and multi-character expressions.",
    },
    {
        "question": "What is vocabulary size in corpus statistics?",
        "ground_truth_answer": "Vocabulary size is the number of distinct word types, often denoted as |V|.",
    },
    {
        "question": "What is represented by N in corpus statistics?",
        "ground_truth_answer": "N denotes the number of running word instances (tokens in the corpus-count sense).",
    },
    {
        "question": "Why is tokenization considered a first step in NLP?",
        "ground_truth_answer": "Most downstream NLP methods operate on token units, so text must be segmented before modeling.",
    },
    {
        "question": "Give one application where edit distance is important.",
        "ground_truth_answer": "Edit distance is important in automatic speech recognition metrics such as word error rate.",
    },
]



In [7]:
@dataclass
class ChunkRecord:
    chunk_id: int
    text: str
    contextual_text: str


class AssignmentRAG:
    def __init__(self, embedding_model: str = "sentence-transformers/all-MiniLM-L6-v2"):
        self.embedding_model_name = embedding_model
        self.embedder = SentenceTransformer(embedding_model)
        self.records: List[ChunkRecord] = []
        self.naive_embeddings = None
        self.contextual_embeddings = None

    def load_pdf(self, path: Path) -> str:
        reader = PdfReader(str(path))
        text = "\n".join((p.extract_text() or "") for p in reader.pages)
        text = text.replace("\u00ad", "")
        text = re.sub(r"\s+", " ", text).strip()
        return text

    def chunk_text(self, text: str, chunk_size: int = 900, overlap: int = 150) -> List[str]:
        words = text.split()
        chunks = []
        i = 0
        while i < len(words):
            chunk = " ".join(words[i:i+chunk_size])
            if chunk:
                chunks.append(chunk)
            if i + chunk_size >= len(words):
                break
            i += (chunk_size - overlap)
        return chunks

    def make_context(self, chunk: str) -> str:
        lower = chunk.lower()
        if "token" in lower:
            topic = "tokenization and segmentation"
        elif "edit distance" in lower:
            topic = "edit distance and string similarity"
        elif "regular expression" in lower or "regex" in lower:
            topic = "regular expressions for text processing"
        elif "byte-pair" in lower or "bpe" in lower:
            topic = "BPE subword tokenization"
        elif "word type" in lower or "word instance" in lower:
            topic = "word types and word instances"
        else:
            topic = "core chapter concepts"

        prefix = f"This chunk from Chapter 2 discusses {topic} in relation to the whole chapter."
        return f"{prefix}\n\n{chunk}"

    def fit(self, chapter_text: str):
        chunks = self.chunk_text(chapter_text)
        self.records = [
            ChunkRecord(chunk_id=i, text=c, contextual_text=self.make_context(c))
            for i, c in enumerate(chunks)
        ]
        naive_texts = [r.text for r in self.records]
        contextual_texts = [r.contextual_text for r in self.records]
        self.naive_embeddings = self.embedder.encode(naive_texts, normalize_embeddings=True)
        self.contextual_embeddings = self.embedder.encode(contextual_texts, normalize_embeddings=True)

    def retrieve(self, question: str, mode: str = "naive", top_k: int = 3) -> List[ChunkRecord]:
        q = self.embedder.encode([question], normalize_embeddings=True)[0]
        matrix = self.naive_embeddings if mode == "naive" else self.contextual_embeddings
        scores = matrix @ q
        idxs = np.argsort(scores)[::-1][:top_k]
        return [self.records[i] for i in idxs]

    def generate_extractive(self, question: str, retrieved: List[ChunkRecord]) -> str:
        context = " ".join(r.text for r in retrieved)
        sents = re.split(r"(?<=[.!?])\s+", context)
        q_terms = set(re.findall(r"[a-zA-Z0-9]+", question.lower()))
        best_sent, best_score = "", -1
        for s in sents:
            s_terms = set(re.findall(r"[a-zA-Z0-9]+", s.lower()))
            score = len(q_terms & s_terms)
            if score > best_score:
                best_score = score
                best_sent = s
        return best_sent.strip() if best_sent else retrieved[0].text[:260]

    def answer(self, question: str, mode: str = "naive", top_k: int = 3) -> Tuple[str, List[ChunkRecord]]:
        retrieved = self.retrieve(question, mode=mode, top_k=top_k)
        return self.generate_extractive(question, retrieved), retrieved


def rouge_avg(preds: List[str], refs: List[str]) -> Dict[str, float]:
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    r1, r2, rl = [], [], []
    for p, r in zip(preds, refs):
        s = scorer.score(r, p)
        r1.append(s["rouge1"].fmeasure)
        r2.append(s["rouge2"].fmeasure)
        rl.append(s["rougeL"].fmeasure)
    return {
        "rouge1": float(np.mean(r1)),
        "rouge2": float(np.mean(r2)),
        "rougeL": float(np.mean(rl)),
    }


print(f"QA pairs loaded: {len(qa_pairs)}")

QA pairs loaded: 20


In [8]:
rag = AssignmentRAG()
chapter_text = rag.load_pdf(PDF_PATH)
rag.fit(chapter_text)

naive_preds = []
ctx_preds = []
refs = []
rows = []

for item in qa_pairs:
    q = item["question"]
    gt = item["ground_truth_answer"]

    naive_ans, naive_chunks = rag.answer(q, mode="naive", top_k=3)
    ctx_ans, ctx_chunks = rag.answer(q, mode="contextual", top_k=3)

    naive_preds.append(naive_ans)
    ctx_preds.append(ctx_ans)
    refs.append(gt)

    rows.append(
        {
            "question": q,
            "ground_truth_answer": gt,
            "naive_rag_answer": naive_ans,
            "contextual_retrieval_answer": ctx_ans,
            "naive_source_chunk_id": naive_chunks[0].chunk_id,
            "naive_source_text": naive_chunks[0].text,
            "contextual_source_chunk_id": ctx_chunks[0].chunk_id,
            "contextual_chunk_before": ctx_chunks[0].text,
            "contextual_chunk_after": ctx_chunks[0].contextual_text,
        }
    )


In [9]:
naive_scores = rouge_avg(naive_preds, refs)
ctx_scores = rouge_avg(ctx_preds, refs)

evaluation_table = [
    {
        "Method": "Naive RAG",
        "ROUGE-1": round(naive_scores["rouge1"], 4),
        "ROUGE-2": round(naive_scores["rouge2"], 4),
        "ROUGE-L": round(naive_scores["rougeL"], 4),
    },
    {
        "Method": "Contextual Retrieval",
        "ROUGE-1": round(ctx_scores["rouge1"], 4),
        "ROUGE-2": round(ctx_scores["rouge2"], 4),
        "ROUGE-L": round(ctx_scores["rougeL"], 4),
    },
]

before_after_example = {
    "question": rows[0]["question"],
    "BEFORE": rows[0]["contextual_chunk_before"],
    "AFTER": rows[0]["contextual_chunk_after"],
}

print("Evaluation Table")
print("Method                  ROUGE-1   ROUGE-2   ROUGE-L")
for row in evaluation_table:
    print(
        f"{row['Method']:<24}{row['ROUGE-1']:.4f}    {row['ROUGE-2']:.4f}    {row['ROUGE-L']:.4f}"
    )

{
    "before_after_example": before_after_example,
    "evaluation_table": evaluation_table,
    "first_result": rows[0],
}

Evaluation Table
Method                  ROUGE-1   ROUGE-2   ROUGE-L
Naive RAG               0.2147    0.0508    0.1523
Contextual Retrieval    0.1962    0.0482    0.1384


{'before_after_example': {'question': 'What is tokenization in modern NLP?',
  'BEFORE': 'tokenization standard is known as the Penn Treebank to- kenization standard, used for the parsed corpora (treebanks) released by the Lin-Penn Treebank tokenization guistic Data Consortium (LDC), the source of many useful datasets. This standard separates out clitics ( doesn’t becomes does plus n’t), keeps hyphenated words to- gether, and separates out all punctuation (to save space we’re showing visible spaces ‘ ’ between tokens, although newlines is a more common output): Input: "The San Francisco-based restaurant," they said, "doesn’t charge $10". Output: " The San Francisco-based restaurant , " they said , " does n’t charge $ 10 " . In practice, since tokenization is run before any other language processing, it needs to be very fast. For rule-based word tokenization we generally use deter- ministic algorithms based on regular expressions compiled into efﬁcient ﬁnite state automata. For example,

## Quick Result Notes

In my run, Naive RAG is slightly higher than Contextual Retrieval.

My reading of why contextual underperformed here:
- Chapter 2 is already focused, so the added context sentence sometimes adds noise instead of helping retrieval.
- I used a heuristic context builder (`make_context`), not an LLM prompt, so some prefixes are too generic.
- My generator is extractive and overlap-based, so if retrieval shifts a little, the final sentence can miss the key fact.

Failure cases I observed:
- **Q: What are filled pauses in spoken language?**
  - Model answer drifted to generic word-type text, so retrieval missed the speech-disfluency part.
- **Q: What is an utterance?**
  - Retrieved ELIZA-related sentence instead of the direct definition.
- **Q: What is represented by N in corpus statistics?**
  - Returned a long BPE training snippet rather than the short token-count definition.

What I would test next:
- Smaller chunk size and overlap tuning.
- LLM-generated context prefixes for each chunk.
- A stronger answer step (reranking or concise generation) after retrieval.

In [10]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

student_digits = "".join(ch for ch in STUDENT_ID if ch.isdigit())
response_file = OUTPUT_DIR / f"response-st-{student_digits}-chapter-{CHAPTER_NO}.json"
metrics_file = OUTPUT_DIR / f"metrics-st-{student_digits}-chapter-{CHAPTER_NO}.json"

with response_file.open("w", encoding="utf-8") as f:
    json.dump(rows, f, indent=2, ensure_ascii=False)

metrics = {
    "student_id": STUDENT_ID,
    "chapter": CHAPTER_NO,
    "retriever_model": rag.embedding_model_name,
    "generator_model": "extractive sentence selector",
    "evaluation_table": evaluation_table,
    "scores": {
        "Naive RAG": {
            "ROUGE-1": round(naive_scores["rouge1"], 4),
            "ROUGE-2": round(naive_scores["rouge2"], 4),
            "ROUGE-L": round(naive_scores["rougeL"], 4),
        },
        "Contextual Retrieval": {
            "ROUGE-1": round(ctx_scores["rouge1"], 4),
            "ROUGE-2": round(ctx_scores["rouge2"], 4),
            "ROUGE-L": round(ctx_scores["rougeL"], 4),
        },
    },
}


In [11]:
with metrics_file.open("w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print("Saved:", response_file)
print("Saved:", metrics_file)

{
    "saved_response_file": str(response_file),
    "saved_metrics_file": str(metrics_file),
    "evaluation_table": evaluation_table,
}

Saved: answer\response-st-125982-chapter-2.json
Saved: answer\metrics-st-125982-chapter-2.json


{'saved_response_file': 'answer\\response-st-125982-chapter-2.json',
 'saved_metrics_file': 'answer\\metrics-st-125982-chapter-2.json',
 'evaluation_table': [{'Method': 'Naive RAG',
   'ROUGE-1': 0.2147,
   'ROUGE-2': 0.0508,
   'ROUGE-L': 0.1523},
  {'Method': 'Contextual Retrieval',
   'ROUGE-1': 0.1962,
   'ROUGE-2': 0.0482,
   'ROUGE-L': 0.1384}]}

---
## Evaluation Summary

| Method | ROUGE-1 | ROUGE-2 | ROUGE-L |
|---|---:|---:|---:|
| Naive RAG | 0.2147 | 0.0508 | 0.1523 |
| Contextual Retrieval | 0.1962 | 0.0482 | 0.1384 |

Naive RAG performed better than Contextual Retrieval on all three evaluation metrics for Chapter 2.

- ROUGE-1 improved by 0.0185
- ROUGE-2 improved by 0.0026
- ROUGE-L improved by 0.0139

Overall, the contextual retrieval variant did not improve overlap with the ground-truth answers in this run, while Naive RAG produced slightly stronger results across all reported ROUGE scores.